Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Train

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# ==== CONFIG ====
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

NUM_SUBJECTS = 123
NUM_FINGERS = 4
IMAGE_SIZE = (100, 300)
TRAIN_IMAGES = [1, 2, 3]

train_images = []
train_labels = []

# ==== STEP 1: LOAD TRAINING DATA ====
def load_training_data_strategy2(base_path, session_label):
    for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc=f"Loading training data - Session {session_label}"):
        for finger_id in range(1, NUM_FINGERS + 1):
            folder_name = f"vein{subject_id:03d}_{finger_id}"
            folder_path = os.path.join(base_path, folder_name)

            for img_idx in TRAIN_IMAGES:
                img_path = os.path.join(folder_path, f"{img_idx:02d}.jpg")
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"❌ Missing: {img_path}")
                    continue

                img = cv2.resize(img, IMAGE_SIZE)
                img_eq = exposure.equalize_hist(img).astype(np.float64)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

                label = f"session{session_label}_subject{subject_id:03d}_finger{finger_id}_img{img_idx:02d}"
                train_images.append(img_norm)
                train_labels.append(label)

                print(f"🟩 Train → {label}")

# ==== APPLY TO BOTH SESSIONS ====
load_training_data_strategy2(base_path_sess1, session_label=1)
load_training_data_strategy2(base_path_sess2, session_label=2)

print(f"\n✅ Total training samples: {len(train_images)}")
print(f"✅ Example label: {train_labels[0]}")

# ==== STEP 2: COMPUTE 2DPCA PROJECTION MATRIX ====
def compute_2dpca_projection(images_2d, num_components):
    print("\n⚙️ Computing 2DPCA projection matrix...")
    n = len(images_2d)
    h, w = images_2d[0].shape
    mean_img = sum(images_2d) / n
    G_t = np.zeros((w, w))

    for i, img in enumerate(images_2d):
        A = img - mean_img
        G_t += A.T @ A
        if i < 3:
            print(f"  ➕ Added image {i+1} to covariance matrix")

    G_t /= n
    eig_vals, eig_vecs = np.linalg.eigh(G_t)
    idx = np.argsort(-eig_vals)
    eig_vecs = eig_vecs[:, idx[:num_components]]
    print(f"✅ Computed projection matrix W shape: {eig_vecs.shape}")
    return eig_vecs

# ==== STEP 3: PROJECT TRAINING IMAGES ====
num_components = 137
W = compute_2dpca_projection(train_images, num_components)

projected_train_features = []
for i, img in enumerate(train_images):
    feat = img @ W
    projected_train_features.append(feat)
    if i < 3:
        print(f"🧮 Projected train sample {i+1} shape: {feat.shape}")

# ==== STEP 4: FLATTEN FEATURES FOR CLASSIFIER ====
flat_train_features = np.array([f.flatten() for f in projected_train_features])
train_labels = np.array(train_labels)

print(f"\n✅ Flattened feature matrix shape: {flat_train_features.shape}")
print(f"🧾 Total training labels: {len(train_labels)}")


Test

In [ ]:
test_data = []
test_labels = []
test_paths = []

TEST_IMAGES = [4, 5, 6]

for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc="Preparing test data"):
    for session_label, base_path in [(1, base_path_sess1), (2, base_path_sess2)]:
        for finger_id in range(1, NUM_FINGERS + 1):
            folder_name = f"vein{subject_id:03d}_{finger_id}"
            folder_path = os.path.join(base_path, folder_name)

            for img_idx in TEST_IMAGES:
                img_path = os.path.join(folder_path, f"{img_idx:02d}.jpg")

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"⚠️ Missing: {img_path}")
                    continue

                print(f"✅ Using: {img_path}")
                img = cv2.resize(img, IMAGE_SIZE)
                img_eq = exposure.equalize_hist(img).astype(np.float64)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

                label = f"session{session_label}_subject{subject_id:03d}_finger{finger_id}_img{img_idx:02d}"
                test_data.append(img_norm)
                test_labels.append(label)
                test_paths.append(img_path)

# ==== Convert to arrays ====
test_data = np.array(test_data)
test_labels = np.array(test_labels)

# ==== STEP X: PROJECT TEST IMAGES USING 2DPCA ====
proj_test_features = [img @ W for img in test_data]
flat_test_features = np.array([f.flatten() for f in proj_test_features])

print(f"\n✅ Projected test features shape: {flat_test_features.shape}")
print(f"🧾 Total test labels: {len(test_labels)}")


Benchmarking

In [ ]:
correct_matches = 0
total_tests = len(flat_test_features)

print("\n🔍 Starting classification using Manhattan distance (match: subject + finger)...")

for i in range(total_tests):
    test_vec = flat_test_features[i]
    true_label = test_labels[i]  # e.g., "session1_subject005_finger2_img06"

    # Compute Manhattan distances to training vectors
    distances = np.sum(np.abs(flat_train_features - test_vec), axis=1)

    # Find nearest neighbor
    min_index = np.argmin(distances)
    predicted_label = train_labels[min_index]  # e.g., "session2_subject005_finger2_img03"

    # Extract subject and finger ID
    def extract_subject_and_finger(label):
        parts = label.split('_')
        subject = parts[1]  # e.g., subject005
        finger = parts[2]   # e.g., finger2
        return subject, finger

    true_subject, true_finger = extract_subject_and_finger(true_label)
    pred_subject, pred_finger = extract_subject_and_finger(predicted_label)

    # Check match
    if true_subject == pred_subject and true_finger == pred_finger:
        correct_matches += 1
        match_result = "✅ MATCH (subject + finger)"
    else:
        match_result = "❌ MISMATCH"

    # Log result
    print(f"\n🧪 Test sample {i+1}/{total_tests}")
    print(f"  🎯 Predicted → {predicted_label}")
    print(f"  ✅ Actual    → {true_label}")
    print(f"  ➡️  Result    → {match_result}")

# Final accuracy
accuracy = (correct_matches / total_tests) * 100
print(f"\n📊 Final Results")
print(f"✅ Correct matches: {correct_matches} / {total_tests}")
print(f"🎯 Recognition Accuracy (Subject + Finger): {accuracy:.2f}%")
